# Comprobar referencias después de un cálculo

El cálculo independiente produce media 2.5 y varianza poblacional 1.25.
Inspeccionamos sus referencias con Recorda y un índice local explícito, retiramos
una entrada y modificamos bytes de una salida, y restauramos ambos archivos.

El estado de ejecución se conserva. Un hash coincidente comprueba bytes locales;
no certifica identidad científica, autoría, retención permanente o replay.
Una referencia sin digest, una que no se puede resolver, una omisión por política
y una operación interrumpida son situaciones distintas. Los índices permanecen
fuera del journal y las bibliotecas científicas conservan sus objetos.


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("experiments").resolve()))
import recorda
from run_reference_checks import prepare, run

workspace = Path("reference-example")
print(sys.executable)

In [ ]:
trial = prepare(workspace)
files = trial["files"]
record = trial["records"]["full"]
assert trial["scientific_result"]["mean"] == 2.5
assert trial["scientific_result"]["variance"] == 1.25

In [ ]:
intact = recorda.check_references(record, resolver=files)
assert len(intact["references"]) == 4
assert all(row["status"] == "matched" for row in intact["references"])
[(row["group"], row["field"], row["status"]) for row in intact["references"]]

In [ ]:
source_path = workspace / "native/samples.json"
source_bytes = source_path.read_bytes()
source_path.unlink()
missing = recorda.check_references(record, resolver=files)
assert {row["status"] for row in missing["references"]} == {"missing", "matched"}
assert missing["session_status"] == "succeeded"
[(row["group"], row["field"], row["status"]) for row in missing["references"]]

In [ ]:
source_path.write_bytes(source_bytes)
result_path = workspace / "native/result.json"
result_bytes = result_path.read_bytes()
result_path.write_bytes(result_bytes + b"\n")
modified = recorda.check_references(record, resolver=files)
assert {row["status"] for row in modified["references"]} == {"mismatched", "matched"}
[(row["group"], row["field"], row["status"]) for row in modified["references"]]

In [ ]:
result_path.write_bytes(result_bytes)
restored = recorda.check_references(record, resolver=files)
assert all(row["status"] == "matched" for row in restored["references"])
assert record.status == "succeeded"

In [ ]:
limits = recorda.check_references(trial["records"]["limits"], resolver=files)
minimal = recorda.check_references(trial["records"]["minimal"], resolver=files)
interrupted = recorda.check_references(trial["records"]["interrupted"], resolver=files)
assert {row["status"] for row in limits["references"]} == {"available_unverified", "unresolved"}
assert minimal["references"] == [] and len(minimal["omissions"]) == 6
assert interrupted["session_status"] == "incomplete" and interrupted["incomplete_operations"]
{
    "reference_limits": [row["status"] for row in limits["references"]],
    "omissions": minimal["omissions"],
    "incomplete": interrupted["incomplete_operations"],
}

In [ ]:
acceptance = run(workspace / "checked-simulation")
assert acceptance["scientific_result"] == trial["scientific_result"]
list(acceptance["reports"])